# Minimal Model Comparison: Quick Execution Verification

**Purpose:** Verify that LogisticGLM and GRU models execute correctly with the utility function.

**Runtime:** ~2-3 minutes (vs 10-15 minutes for full test)

**Configuration:**
- Models: LogisticGLM, GRU (skipping XGBoost to reduce dependencies)
- Training sizes: [50, 100]
- Bootstrap sizes: [25]
- Iterations: 3 per config
- **Total: 2 × 2 × 1 × 3 = 12 evaluations**

After verifying execution, use `test_utility_model_comparison.ipynb` for full evaluation with all models and configurations.

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Add rebuild directory to path
sys.path.insert(0, str(Path.cwd()))

from data_loader import load_data, add_hours_until_sepsis
from models import LogisticGLMModel, GRUModel
from training import BootstrapEvaluator
from utility import physionet_utility

print("✓ All imports successful")

## 1. Configuration

Minimal experimental grid for quick verification:

In [ ]:
# MINIMAL configuration for quick execution
TRAIN_SIZES = [50, 100]        # 2 training sizes (skip 200, 300)
BOOTSTRAP_SIZES = [25]         # 1 bootstrap size (skip 50, 100)
N_ITER = 3                     # 3 iterations per config (vs 5 in full)
RANDOM_STATE = 42

# Model configurations
MODEL_CONFIGS = {
    'LogisticGLM': {'C': 0.01, 'penalty': 'l1', 'solver': 'saga', 'max_iter': 10000},
    'GRU': {'hidden_size': 64, 'num_layers': 1, 'dropout': 0.2, 'epochs': 10}  # Reduced from 20 for speed
}

print(f"Training sizes: {TRAIN_SIZES}")
print(f"Bootstrap sizes: {BOOTSTRAP_SIZES}")
print(f"Iterations per config: {N_ITER}")
print(f"Total evaluations: {len(TRAIN_SIZES)} × {len(BOOTSTRAP_SIZES)} × {N_ITER} = {len(TRAIN_SIZES) * len(BOOTSTRAP_SIZES) * N_ITER}")

## 2. Load Data

In [ ]:
# Load patient data
data_dir = Path('../data/processed')
patients = []
patient_list = sorted([p for p in data_dir.glob('p*.pkl') if p.is_file()])[:200]  # Limit to first 200 for speed

print(f"Loading {len(patient_list)} patient files...")
for pfile in patient_list:
    try:
        p = load_data(str(pfile))
        if p is not None and len(p) > 10:  # Minimum sequence length
            patients.append(p)
    except Exception as e:
        print(f"Skipped {pfile.name}: {e}")

print(f"✓ Loaded {len(patients)} valid patients")

# Display sample patient
if patients:
    sample = patients[0]
    print(f"\nSample patient shape: {sample.shape}")
    print(f"Columns: {list(sample.columns)}")
    print(f"\nFirst few rows:\n{sample.head()}")

## 3. Run Minimal Evaluation Grid

Quick iteration through 2 models × 2 train sizes × 1 boot size × 3 iterations = 12 total runs

In [ ]:
results = []
config_num = 0
total_configs = len(TRAIN_SIZES) * len(BOOTSTRAP_SIZES) * N_ITER

for model_name in MODEL_CONFIGS.keys():
    print(f"\n{'='*70}")
    print(f"MODEL: {model_name}")
    print(f"{'='*70}")
    
    for train_size in TRAIN_SIZES:
        print(f"\n  Train size: {train_size}")
        
        for boot_size in BOOTSTRAP_SIZES:
            print(f"    Boot size: {boot_size}")
            
            for iter_idx in range(N_ITER):
                config_num += 1
                print(f"      [{config_num}/{total_configs}] Iteration {iter_idx+1}... ", end='', flush=True)
                
                try:
                    # Create model
                    if model_name == 'LogisticGLM':
                        model = LogisticGLMModel(**MODEL_CONFIGS[model_name])
                    elif model_name == 'GRU':
                        model = GRUModel(**MODEL_CONFIGS[model_name])
                    
                    # Run bootstrap evaluation
                    evaluator = BootstrapEvaluator(
                        patients=patients,
                        model=model,
                        train_size=train_size,
                        bootstrap_size=boot_size,
                        label_column='SepsisLabel',
                        random_state=RANDOM_STATE + config_num
                    )
                    
                    metrics = evaluator.evaluate()
                    
                    # Store results
                    results.append({
                        'model': model_name,
                        'train_size': train_size,
                        'boot_size': boot_size,
                        'iteration': iter_idx,
                        'utility': metrics['utility'],
                        'auroc': metrics.get('auroc', np.nan),
                        'recall': metrics.get('recall', np.nan),
                        'f1': metrics.get('f1', np.nan),
                        'utility_female': metrics.get('utility_female', np.nan),
                        'utility_male': metrics.get('utility_male', np.nan),
                    })
                    
                    print(f"✓ Utility: {metrics['utility']:.4f}")
                    
                except Exception as e:
                    print(f"✗ ERROR: {e}")
                    results.append({
                        'model': model_name,
                        'train_size': train_size,
                        'boot_size': boot_size,
                        'iteration': iter_idx,
                        'utility': np.nan,
                        'auroc': np.nan,
                        'recall': np.nan,
                        'f1': np.nan,
                        'utility_female': np.nan,
                        'utility_male': np.nan,
                    })

print(f"\n\n{'='*70}")
print(f"EVALUATION COMPLETE")
print(f"{'='*70}")

## 4. Results Summary

In [ ]:
# Create results dataframe
results_df = pd.DataFrame(results)

print(f"\nTotal results collected: {len(results_df)}")
print(f"\nResults dataframe shape: {results_df.shape}")
print(f"\nFirst 10 rows:\n")
print(results_df.head(10))

## 5. Best Configurations by Model

In [ ]:
print("\nBEST CONFIGURATIONS BY MODEL")
print("="*80)

for model_name in results_df['model'].unique():
    model_results = results_df[results_df['model'] == model_name]
    best_idx = model_results['utility'].idxmax()
    best = model_results.loc[best_idx]
    
    print(f"\n{model_name}:")
    print(f"  Train size: {best['train_size']:.0f} patients")
    print(f"  Boot size: {best['boot_size']:.0f} patients/sample")
    print(f"  Utility: {best['utility']:.4f}")
    print(f"  AUROC: {best['auroc']:.4f}")
    print(f"  Recall: {best['recall']:.4f}")
    print(f"  F1: {best['f1']:.4f}")
    print(f"  Gender utility gap: {abs(best['utility_female'] - best['utility_male']):.4f}")

## 6. Aggregated Summary

In [ ]:
# Summary by model and train size
summary_df = results_df.groupby(['model', 'train_size']).agg({
    'utility': ['mean', 'std', 'min', 'max'],
    'auroc': ['mean', 'std'],
    'recall': ['mean', 'std'],
    'f1': ['mean', 'std']
}).round(4)

print("\nSUMMARY BY MODEL AND TRAINING SIZE")
print("="*80)
print(summary_df)

# Save summary
summary_df.to_csv('utility_model_comparison_summary_minimal.csv')
print("\n✓ Summary saved to utility_model_comparison_summary_minimal.csv")

## 7. Visualizations

In [ ]:
# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)

# 1. Utility by Model and Training Size
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Utility
for model in results_df['model'].unique():
    model_data = results_df[results_df['model'] == model]
    train_sizes = sorted(model_data['train_size'].unique())
    utilities = [model_data[model_data['train_size'] == ts]['utility'].mean() for ts in train_sizes]
    axes[0].plot(train_sizes, utilities, marker='o', label=model, linewidth=2, markersize=8)

axes[0].set_xlabel('Training Size (patients)', fontsize=12)
axes[0].set_ylabel('Mean Utility', fontsize=12)
axes[0].set_title('Utility Score vs Training Size', fontsize=13, fontweight='bold')
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)

# AUROC
for model in results_df['model'].unique():
    model_data = results_df[results_df['model'] == model]
    train_sizes = sorted(model_data['train_size'].unique())
    aurocs = [model_data[model_data['train_size'] == ts]['auroc'].mean() for ts in train_sizes]
    axes[1].plot(train_sizes, aurocs, marker='s', label=model, linewidth=2, markersize=8)

axes[1].set_xlabel('Training Size (patients)', fontsize=12)
axes[1].set_ylabel('Mean AUROC', fontsize=12)
axes[1].set_title('AUROC vs Training Size', fontsize=13, fontweight='bold')
axes[1].legend(fontsize=11)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('model_comparison_metrics_minimal.png', dpi=150, bbox_inches='tight')
print("✓ Saved: model_comparison_metrics_minimal.png")
plt.show()

In [ ]:
# 2. Distribution Box Plots
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Utility distributions
sns.boxplot(data=results_df, x='model', y='utility', ax=axes[0])
axes[0].set_title('Utility Score Distributions', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Utility', fontsize=12)
axes[0].set_xlabel('Model', fontsize=12)

# AUROC distributions
sns.boxplot(data=results_df, x='model', y='auroc', ax=axes[1])
axes[1].set_title('AUROC Distributions', fontsize=13, fontweight='bold')
axes[1].set_ylabel('AUROC', fontsize=12)
axes[1].set_xlabel('Model', fontsize=12)

plt.tight_layout()
plt.savefig('model_distributions_minimal.png', dpi=150, bbox_inches='tight')
print("✓ Saved: model_distributions_minimal.png")
plt.show()

In [ ]:
# 3. Gender Fairness Gap
fig, ax = plt.subplots(figsize=(10, 5))

results_df['gender_gap'] = (results_df['utility_female'] - results_df['utility_male']).abs()

fairness_data = results_df.groupby(['model', 'train_size'])['gender_gap'].mean().reset_index()

pivot = fairness_data.pivot(index='model', columns='train_size', values='gender_gap')
pivot.plot(kind='bar', ax=ax, width=0.7)

ax.set_title('Gender Utility Gap by Model and Training Size', fontsize=13, fontweight='bold')
ax.set_xlabel('Model', fontsize=12)
ax.set_ylabel('|Utility (Female) - Utility (Male)|', fontsize=12)
ax.legend(title='Train Size', fontsize=10)
ax.grid(True, alpha=0.3, axis='y')
plt.xticks(rotation=0)

plt.tight_layout()
plt.savefig('fairness_gender_gap_minimal.png', dpi=150, bbox_inches='tight')
print("✓ Saved: fairness_gender_gap_minimal.png")
plt.show()

## 8. Save Results CSV

In [ ]:
# Save full results
results_df.to_csv('utility_model_comparison_results_minimal.csv', index=False)

print("\n" + "="*70)
print("EXECUTION VERIFICATION COMPLETE")
print("="*70)
print(f"\n✓ Total evaluations completed: {len(results_df)}")
print(f"✓ Models tested: {', '.join(results_df['model'].unique())}")
print(f"✓ CSV results saved: utility_model_comparison_results_minimal.csv")
print(f"✓ Summary saved: utility_model_comparison_summary_minimal.csv")
print(f"\nFigures generated:")
print("  - model_comparison_metrics_minimal.png")
print("  - model_distributions_minimal.png")
print("  - fairness_gender_gap_minimal.png")
print(f"\nNext: Run test_utility_model_comparison.ipynb with full config for production results.")
print("="*70)